# BioMQM — Prompt Ablation Study

Full pipeline for the prompt ablation study across **3 strategies** × **5 languages**:
1. **Source QA** — Run QA with P1-fewshot, P2-cot, P3-concise
2. **BT QA** — Run QA on back-translated text for each language
3. **CoT Cleaning** — Clean verbose CoT answers
4. **Mapping** — Combine source and BT answers per strategy
5. **String Comparison** — F1, EM, chrF, BLEU evaluation
6. **SBERT** — Cosine similarity evaluation

| Strategy | Description |
|----------|-------------|
| P1-fewshot | Few-shot examples in prompt |
| P2-cot | Chain-of-thought reasoning |
| P3-concise | Short, direct answers |

## 0. Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(DRIVE_CACHE_DIR, 'sentence_transformers')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate', 'nltk',
                'sentence-transformers', 'sacrebleu', 'textstat',
                'pandas', 'matplotlib', 'seaborn'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

## 1. Pre-download Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import torch

# Qwen
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct',
                                             torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer

# SBERT
sbert_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
del sbert_model

torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Models cached!')

## 2. Path Configuration

In [ ]:
# ============================================
# PATH CONFIGURATION
# ============================================

ABLATION_DIR = f"{PROJECT_ROOT}/Qwen2.5-3B-Instruct/biomqm/prompt-ablation"
CODE_DIR = f"{ABLATION_DIR}/code"
EVAL_DIR = f"{ABLATION_DIR}/evaluation"

# Baseline QG output (input for QA)
QG_PATH = f"{PROJECT_ROOT}/Qwen2.5-3B-Instruct/biomqm/baseline/QG/qwen-3b.jsonl"
ORIGINAL_DATASET = f"{PROJECT_ROOT}/biomqm/dev_with_backtranslation.jsonl"

STRATEGIES = ['P1-fewshot', 'P2-cot', 'P3-concise']
LANGUAGES = ['de', 'es', 'fr', 'ru', 'zh-CN']

# Create output directories
for strategy in STRATEGIES:
    os.makedirs(f'{ABLATION_DIR}/QA/{strategy}', exist_ok=True)
    os.makedirs(f'{ABLATION_DIR}/QA/{strategy}/clean', exist_ok=True)
    os.makedirs(f'{ABLATION_DIR}/{strategy}/evaluation/sbert', exist_ok=True)
    os.makedirs(f'{ABLATION_DIR}/{strategy}/evaluation/string-comparison', exist_ok=True)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f'ABLATION_DIR: {ABLATION_DIR}')
print(f'QG input: {QG_PATH}')

## 3. Source QA (All Strategies)

In [ ]:
for strategy in STRATEGIES:
    output_file = f'{ABLATION_DIR}/QA/{strategy}/source-{strategy}.jsonl'
    cmd = [sys.executable, '-u', f'{CODE_DIR}/qa_ablation.py',
           '--strategy', strategy,
           '--mode', 'source',
           '--qg_input_path', QG_PATH,
           '--output_path', output_file]

    print(f'\nRunning Source QA [{strategy}]...')
    subprocess.run(cmd, check=True)
    print(f'✓ Source QA [{strategy}] complete!')

print('\n✓ All Source QA runs complete!')

## 4. BT QA (All Strategies × All Languages)

In [ ]:
for strategy in STRATEGIES:
    for lang in LANGUAGES:
        output_file = f'{ABLATION_DIR}/QA/{strategy}/bt-{lang}-{strategy}.jsonl'
        cmd = [sys.executable, '-u', f'{CODE_DIR}/qa_ablation.py',
               '--strategy', strategy,
               '--mode', 'bt',
               '--lang', lang,
               '--qg_input_path', QG_PATH,
               '--output_path', output_file]

        print(f'Running BT QA [{strategy}] [{lang}]...')
        subprocess.run(cmd, check=True)
        print(f'✓ BT QA [{strategy}] [{lang}] complete!')

print('\n✓ All BT QA runs complete!')

## 5. CoT Answer Cleaning

P2-CoT produces verbose reasoning chains. Clean them to extract final answers only.

In [ ]:
clean_script = f'{ABLATION_DIR}/code/clean_cot_answers.py'

if os.path.exists(clean_script):
    print('Running CoT answer cleaning...')
    subprocess.run([sys.executable, '-u', clean_script], check=True)
    print('✓ CoT cleaning complete!')
else:
    # Fallback: try the script in the ablation root
    clean_script_alt = f'{ABLATION_DIR}/clean_cot_answers.py'
    if os.path.exists(clean_script_alt):
        print('Running CoT answer cleaning...')
        subprocess.run([sys.executable, '-u', clean_script_alt], check=True)
        print('✓ CoT cleaning complete!')
    else:
        print('⚠ clean_cot_answers.py not found, skipping CoT cleaning')

## 6. Mapping (All Strategies)

Combines source and BT answers per strategy. Prioritizes cleaned CoT files when available.

In [ ]:
# Import mapping function
mapping_script = f'{ABLATION_DIR}/mapping.py'
if os.path.exists(mapping_script):
    # If mapping.py exists as a standalone script, use subprocess
    for strategy in STRATEGIES:
        mapping_output = f'{ABLATION_DIR}/{strategy}/mapping.jsonl'
        os.makedirs(os.path.dirname(mapping_output), exist_ok=True)
        
        # Determine source file (prefer clean)
        clean_src = f'{ABLATION_DIR}/QA/{strategy}/clean/clean-source-{strategy}.jsonl'
        raw_src = f'{ABLATION_DIR}/QA/{strategy}/source-{strategy}.jsonl'
        qa_source = clean_src if os.path.exists(clean_src) else raw_src
        
        # Determine BT files (prefer clean)
        bt_files = []
        for lang in LANGUAGES:
            clean_bt = f'{ABLATION_DIR}/QA/{strategy}/clean/clean-bt-{lang}-{strategy}.jsonl'
            raw_bt = f'{ABLATION_DIR}/QA/{strategy}/bt-{lang}-{strategy}.jsonl'
            bt_files.append(clean_bt if os.path.exists(clean_bt) else raw_bt)
        
        print(f'Running Mapping [{strategy}]...')
        print(f'  Source: {os.path.basename(qa_source)}')
        for f in bt_files:
            print(f'  BT: {os.path.basename(f)}')
        
        cmd = [sys.executable, '-u', mapping_script,
               '--strategy', strategy,
               '--qa_source_path', qa_source,
               '--qa_bt_dir', os.path.dirname(bt_files[0]),
               '--qg_input_path', QG_PATH,
               '--output_path', mapping_output]
        subprocess.run(cmd, check=True)
        print(f'✓ Mapping [{strategy}] complete!')
else:
    print('⚠ mapping.py not found — run mapping manually')

print('\n✓ All Mapping runs complete!')

## 7. String Comparison (All Strategies)

In [ ]:
for strategy in STRATEGIES:
    mapping_file = f'{ABLATION_DIR}/{strategy}/mapping.jsonl'
    output_dir = f'{ABLATION_DIR}/{strategy}/evaluation/string-comparison'
    sc_script = f'{EVAL_DIR}/string_comparison.py'

    cmd = [sys.executable, '-u', sc_script,
           '--input_path', mapping_file,
           '--output_dir', output_dir]

    print(f'Running String Comparison [{strategy}]...')
    subprocess.run(cmd, check=True)
    print(f'✓ String Comparison [{strategy}] complete!')

print('\n✓ All String Comparison runs complete!')

## 8. SBERT Evaluation (All Strategies)

In [ ]:
for strategy in STRATEGIES:
    mapping_file = f'{ABLATION_DIR}/{strategy}/mapping.jsonl'
    output_dir = f'{ABLATION_DIR}/{strategy}/evaluation/sbert'
    sbert_script = f'{EVAL_DIR}/sbert.py'

    cmd = [sys.executable, '-u', sbert_script,
           '--input_path', mapping_file,
           '--output_dir', output_dir]

    print(f'Running SBERT [{strategy}]...')
    subprocess.run(cmd, check=True)
    print(f'✓ SBERT [{strategy}] complete!')

print('\n✓ All SBERT evaluations complete!')

## Summary

Pipeline complete! Output structure:
```
prompt-ablation/
├── QA/{P1-fewshot,P2-cot,P3-concise}/
│   ├── source-{strategy}.jsonl
│   ├── bt-{lang}-{strategy}.jsonl
│   └── clean/   (cleaned CoT files)
├── {P1-fewshot,P2-cot,P3-concise}/
│   ├── mapping.jsonl
│   └── evaluation/
│       ├── sbert/
│       └── string-comparison/
```